<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/May31_Shreyans_gemini_basic_chain_of_thought_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install langchain==0.3.27 langchain-community==0.3.24 langchain-google-genai

In [2]:
# check langchain version it should be 0.3 if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

0.3.27
0.3.24


In [3]:
import os
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Configure llm via .env

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)

In [5]:
# Define reasoning templates for different problem types
REASONING_TEMPLATES = {
    "math_word_problem": """
Let's think through this step by step:
1. First, identify what the problem is asking for
2. Extract the relevant numbers and variables
3. Determine the mathematical operations needed
4. Set up the equation or calculation
5. Solve step by step
6. Verify the answer makes sense
""",

    "logical_deduction": """
Let's reason through this logically:
1. List all the given facts and constraints
2. Identify relationships between elements
3. Look for contradictions or implications
4. Make inferences from the available information
5. Build conclusions step by step
6. Check for consistency
""",

    "code_debugging": """
Let's debug this systematically:
1. Understand what the code should do vs what it actually does
2. Identify the specific error or unexpected behavior
3. Trace through the code execution step by step
4. Check variable values and data flow
5. Isolate the problematic section
6. Propose and test fixes
"""
}


In [6]:
# Example problems - you can replace these dynamically
problems = {
    "math_problem": "If a store has 15 apples and sells 8, then receives a shipment of 12 more apples, how many apples do they have now?",

    "logic_puzzle": "There are three people: Alice, Bob, and Charlie. Alice is taller than Bob. Charlie is shorter than Bob. Who is the tallest?",

    "debugging_issue": "This Python function should return the sum of even numbers in a list, but it's returning incorrect results:\n\n```python\ndef sum_even_numbers(numbers):\n    total = 0\n    for num in numbers:\n        if num % 2 == 0:\n            total += num\n    return total\n\n# Test case: sum_even_numbers([1, 2, 3, 4, 5, 6]) returns 9 instead of 12\n```"
}


In [7]:
# Select a problem to solve
selected_problem = problems["math_problem"]
problem_type = "math_word_problem"


In [8]:
# Build the chain of thought prompt
prompt = f"""
You are an AI reasoning assistant. Your task is to solve problems using clear, step-by-step chain of thought reasoning.

[PROBLEM]
{selected_problem}

[REASONING APPROACH]
{REASONING_TEMPLATES[problem_type]}

Please work through the problem systematically, showing your reasoning at each step before providing the final answer.

Format your response as:
Step 1: [Your first reasoning step]
Step 2: [Your next reasoning step]
...
Final Answer: [Your conclusive answer]
"""


In [9]:
from langchain.schema import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=(
        "You are a clear, careful assistant. Do NOT reveal private chain-of-thought. "
        "Provide a concise final answer and a brief, high-level, non-sensitive stepwise summary "
        "explaining the approach (3–8 numbered steps). Keep the internal detailed reasoning private."
    )),
    HumanMessage(content=prompt),
]

# Call the model
response = llm(messages)  # returns a ChatMessage object
result_text = response.content

# Print result
print("\n--- MODEL OUTPUT ---\n")
print(result_text)


--- MODEL OUTPUT ---

Step 1: Start with the initial number of apples the store has.
Step 2: Subtract the number of apples that were sold to find the remaining quantity.
Step 3: Add the number of apples received in the shipment to the remaining quantity.
Step 4: Calculate the final total.

Final Answer: The store has 19 apples now.


# Alternative prompt for more complex multi-step reasoning


In [10]:
complex_prompt = f"""
You are an expert problem solver. Use chain of thought reasoning to break down this problem into manageable steps.

Problem: {selected_problem}

Think through this carefully:

First, let me understand what's being asked:
[Your initial analysis]

Now, let me identify the key information:
[Extract relevant facts and data]

Next, I'll determine the approach:
[Outline your solution strategy]

Then, I'll work through the steps:
[Detailed step-by-step reasoning]

Finally, I'll verify the solution:
[Check your work and confirm]

Answer:
"""

In [11]:
from langchain.schema import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=(
        "You are a clear, careful assistant. Do NOT reveal private chain-of-thought. "
        "Provide a concise final answer and a brief, high-level, non-sensitive stepwise summary "
        "explaining the approach (3–8 numbered steps). Keep the internal detailed reasoning private."
    )),
    HumanMessage(content=complex_prompt),
]

# Call the model
complex_response = llm(messages)  # returns a ChatMessage object
result_text = complex_response.content

# Print result
print("\n--- MODEL OUTPUT ---\n")
print(result_text)


--- MODEL OUTPUT ---

First, let me understand what's being asked:
The problem asks for the total number of apples a store possesses after a sequence of events: starting with a certain amount, selling some, and then receiving more.

Now, let me identify the key information:
*   Initial number of apples: 15
*   Number of apples sold: 8
*   Number of apples received in shipment: 12

Next, I'll determine the approach:
I will calculate the number of apples remaining after the sale, and then add the number of apples from the new shipment to find the final total.

Then, I'll work through the steps:
1.  Start with the initial number of apples: 15.
2.  Subtract the number of apples sold: 15 - 8 = 7 apples.
3.  Add the number of apples received in the shipment: 7 + 12 = 19 apples.

Finally, I'll verify the solution:
The store began with 15. Selling 8 leaves 7. Receiving 12 more brings the total to 19. The calculations are straightforward and the result makes logical sense given the operations.

TO DO: Now try with this SystemMessage(content=: replace below and observe output

"You are a meticulous reasoning assistant that shows your work step by step."

In [12]:
from langchain.schema import SystemMessage, HumanMessage

def reflective_reasoning(problem: str, problem_type: str, llm, temp: float = 0.1, max_tokens: int = 1000):
    """
    Ask the model for a safe, structured explanation + reflection.
    Returns a dict with keys: final_answer, steps_summary, verification, raw_output.
    """
    # System message: explicitly instruct the model NOT to reveal private chain-of-thought
    system = SystemMessage(content=(
        "You are a careful, clear assistant. Do NOT reveal internal/private chain-of-thought. "
        "Provide a short 'Final Answer', a numbered 'Steps (summary)' that explains the approach at a high level "
        "(3-8 concise steps), and a 'Reflection & Verification' section that checks the result and lists simple checks."
    ))

    user_prompt = f"""Problem type: {problem_type}

Problem:
{problem}

Please respond in this exact format:

Final Answer:
- <one or two sentence conclusion>

Steps (summary):
1) ...
2) ...
(keep to 3-8 short numbered steps describing the method — not private thoughts)

Reflection & Verification:
- Brief checks you did or simple ways a learner can verify the answer
- Any assumptions made
"""

    human = HumanMessage(content=user_prompt)

    # call the llm
    response = llm([system, human], temperature=temp, max_tokens=max_tokens)
    raw = response.content.strip()

    # naive parsing: split by section headers
    # (keeps learners' code simple — robust parsing can be added later)
    def extract_section(text, header):
        idx = text.find(header)
        if idx == -1:
            return ""
        # find next header (either "Steps (summary):" or "Reflection & Verification:" or end)
        next_headers = ["Final Answer:", "Steps (summary):", "Reflection & Verification:"]
        # remove current header from search list
        remaining = [h for h in next_headers if h != header]
        next_idx = len(text)
        for h in remaining:
            i = text.find(h, idx + len(header))
            if i != -1 and i < next_idx:
                next_idx = i
        return text[idx + len(header):next_idx].strip()

    final_answer = extract_section(raw, "Final Answer:")
    steps_summary = extract_section(raw, "Steps (summary):")
    verification = extract_section(raw, "Reflection & Verification:")

    return {
        "final_answer": final_answer,
        "steps_summary": steps_summary,
        "verification": verification,
        "raw_output": raw
    }


# Example usage
selected_problem = "A bicyclist travels 30 km at 15 km/h and then 20 km at 10 km/h. What is the average speed for the whole trip?"
problem_type = "algebra / average speed"

result = reflective_reasoning(selected_problem, problem_type, llm)
print("\nFINAL ANSWER:\n", result["final_answer"])
print("\nSTEPS (SUMMARY):\n", result["steps_summary"])
print("\nREFLECTION & VERIFICATION:\n", result["verification"])




FINAL ANSWER:
 The average speed for the whole trip is 12.5 km/h.

STEPS (SUMMARY):
 1) Calculate the time taken for the first part of the trip using the formula time = distance / speed.
2) Calculate the time taken for the second part of the trip using the same formula.
3) Determine the total distance traveled by summing the distances of both parts.
4) Determine the total time taken for the entire trip by summing the times calculated in steps 1 and 2.
5) Calculate the average speed by dividing the total distance by the total time.

REFLECTION & VERIFICATION:
 - The calculated average speed (12.5 km/h) is between the two given speeds (10 km/h and 15 km/h), which is a good indicator that the answer is reasonable.
- Since the bicyclist spent an equal amount of time (2 hours) at each speed, the average speed is simply the arithmetic mean of the two speeds: (15 km/h + 10 km/h) / 2 = 12.5 km/h. This confirms the result.
- No specific assumptions were made beyond the standard definition of a

In [13]:
# If you want the full raw model response (for debugging)
result["raw_output"]


'Final Answer:\nThe average speed for the whole trip is 12.5 km/h.\n\nSteps (summary):\n1) Calculate the time taken for the first part of the trip using the formula time = distance / speed.\n2) Calculate the time taken for the second part of the trip using the same formula.\n3) Determine the total distance traveled by summing the distances of both parts.\n4) Determine the total time taken for the entire trip by summing the times calculated in steps 1 and 2.\n5) Calculate the average speed by dividing the total distance by the total time.\n\nReflection & Verification:\n- The calculated average speed (12.5 km/h) is between the two given speeds (10 km/h and 15 km/h), which is a good indicator that the answer is reasonable.\n- Since the bicyclist spent an equal amount of time (2 hours) at each speed, the average speed is simply the arithmetic mean of the two speeds: (15 km/h + 10 km/h) / 2 = 12.5 km/h. This confirms the result.\n- No specific assumptions were made beyond the standard defin

In [14]:
# Key Features:
# Multiple Reasoning Templates - Different approaches for math, logic, and debugging problems
# Structured Step-by-Step Format - Clear progression through reasoning steps
# Self-Reflection Capability - Includes verification and alternative approach consideration
# Multi-Problem Processing - Can handle sequences of problems with appropriate reasoning styles
# Flexible Prompt Construction - Easy to adapt for different problem types

# Benefits of Chain of Thought:
# Transparent Reasoning - Shows how the solution was reached
# Error Identification - Makes it easier to spot where reasoning goes wrong
# Learning Tool - Demonstrates problem-solving methodology
# Better Accuracy - Step-by-step reasoning often leads to more correct answers
# Debugging Friendly - Easy to follow the logic trail